# Why use fine-tuning 

Fine-tuning updates the model’s 'knowledge' (internal weights) by training it on your own dataset so it specializes in what you need.

![](images/finetuning_cc.png)


Example before Fine-Tuning:

🤖 "What is the refund policy?" → "I'm not sure, but I can help!"

Example after Fine-Tuning:

🤖 "What is the refund policy?" → "Refunds are allowed within 30 days."


<span style="color: green;">**QUESTION**</span>: When should you fine-tune and shouldn't you fine-tune?

<details>

 <summary>Answer: Click to show</summary>

- When the model isn’t specialized enough (e.g., medical, legal, finance) AND 
- prompting doesn’t give reliable results AND 
- you have a large, high-quality dataset AND 
- You have the time and budget

❌ DON’T fine-tune if you can get good results by just improving your prompts! Fine-tuning is expensive and takes time.

</details>

----

# How to fine-tune a model

1. Pick a Pretrained Model - you choose a model that is already good at language.
2. Prepare Your Training Data - your dataset consists of input-output pairs (in CSV, JSON, or text format). 
3. Train the Model - you feed the training data into the model so it can adjust its parameters.
4. Test the Model - after training, you give the model new inputs to see if it learned correctly.
5. Deploy the Model - so that you can use it in your app via an API!


<span style="color: green;">**QUESTION**</span>: Why do we need to find a model that was already trained?

<details>
    <summary>Click to show the answer</summary>

**Starting point**: Pre-trained models have already learned a vast amount of information from large datasets, which provides a strong foundation for fine-tuning. Fine-tuning leverages this existing knowledge to adapt the model to specific tasks, making it more efficient and accurate. The existing knowledge here would be general language understanding, grammar, common knowledge, or even domain-specific knowledge and the "skills" to perform tasks that are similar to the one you want to do.

**Efficiency and Cost**: Fine-tuning a pre-trained model is more cost-effective and faster than training a model from scratch. It reduces the time and computational resources required.

</details>

## Fine-tuning with Hugging Face's Trainer class: 

Transformers provides a Trainer class optimized for training 🤗 Transformers models, making it easier to start training without manually writing your own training loop. The Trainer API supports a wide range of training options and features such as logging, gradient accumulation, and mixed precision.

See https://huggingface.co/docs/transformers//training#train-with-pytorch-trainer 

At its simplest, it consists of something like this: 

```python 
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(output_dir="./results", per_device_train_batch_size=8)
trainer = Trainer(model=model, args=training_args, train_dataset=custom_dataset)
trainer.train()
```

#### Start by loading your model and specify the number of expected labels. 

 By default, the weights are loaded in full precision (`torch.float32`) regardless of the actual data type the weights are stored in such as `torch.float16`. Set torch_dtype="auto" to load the weights in the data type defined in a model’s `config.json` file to automatically load the most memory-optimal data type.


In this lab, we will use the `trl`(https://github.com/huggingface/trl) library to fine-tune the model. This is a generic library that wraps around `transformers` and lets us easily specify how we want to run finetuning.

-----

## Partial vs. full fine-tuning

Choosing between full fine-tuning and partial fine-tuning is a critical decision that depends on the specific requirements of your use case, the size of the model, and the computational resources available. Full fine-tuning is best suited for cases where extensive customization and task-specific adaptation are required, while partial fine-tuning offers a more resource-efficient alternative for achieving good results without sacrificing general knowledge. Below, we outline when each approach is most appropriate.


### Full Fine-Tuning:

- **When to Use:**
  - When the task requires extensive adaptation that cannot be achieved by modifying only a subset of the parameters.
  - For highly specialized applications where the model needs to significantly diverge from its pre-trained knowledge.
- **Advantages:**
  - Allows complete optimization of the model for the target task.
  - Ensures maximal flexibility in adapting the model's behavior.
- **Challenges:**
  - Requires significant GPU resources and time.
  - Requires larger training dataset.
  - Higher risk of overfitting or catastrophic forgetting, particularly for limited training data.

### Partial Fine-Tuning:
- **When to Use:**
  - For tasks that require fine-tuning on resource-constrained hardware or limited GPUs.
  - When maintaining the general knowledge of the pre-trained model is crucial (e.g. domain-specific question answering).
  - For tasks where relatively small adjustments to the model suffice for optimal performance.
- **Advantages:**
  - Requires fewer trainable parameters, which makes it more efficient in terms of memory and compute.
  - Reduces the risk of catastrophic forgetting by freezing most of the pre-trained model's knowledge.
- **Techniques:**
  - Methods like LoRA, QLoRA and DoRA to optimize only key parts of the model:
    - LoRA (Low-Rank Adaptation): Adds small, trainable low-rank matrices to the model for task-specific fine-tuning.
    - QLoRA (Quantized LoRA): Extends LoRA by using quantized weights (lower precision) to reduce memory requirements further while maintaining performance.
    - DoRA (Dynamic LoRA): Dynamically adjusts trainable parameters based on the complexity of the task.

In this lab, we will focus on partial fine-tuning using the LoRA method to optimize only key parts of the model, as it is more efficient in terms of memory and compute resources.